# Solving a UKLO Rosetta puzzle with a logic solver (clingo / ASP)

**Phase 4 spike** — research question 5: *does externalized, verifiable hypothesis search
outperform monolithic chain-of-thought?* Before comparing against LLMs at scale, this notebook
demonstrates the solver side end-to-end on one real puzzle and one real question.

**Puzzle:** [UKLO 2023 Round 1, Problem 3 — Gilbertese](https://www.uklo.org/wp-content/uploads/2023/03/2023_R1_3-Gilbertese.pdf)
(10 marks, Austronesian, VOS word order, object-number agreement on the verb).

**Question (Q 3.1a):** translate into English: **"E noorii taian uaa te aomata."**

**Solver:** [clingo](https://potassco.org/clingo/) (Answer Set Programming). Rationale over
alternatives (Z3, Prolog, ILP systems): the core of a Rosetta puzzle is a *joint combinatorial
search* — align each source token to a semantic element, hypothesize a lexicon
(one form ↔ one meaning), hypothesize a constituent order, and test every hypothesis against
every context pair simultaneously. ASP expresses exactly that as choice rules + integrity
constraints, enumerates **all** consistent grammars (so we can certify uniqueness, which no CoT
can), and grounds/solves this instance in milliseconds.

**Division of labor** (kept honest):

| Step | Who does it | Why |
|---|---|---|
| Parse English glosses into semantic frames | mechanical Python (an LLM in the full pipeline) | English is the metalanguage; no puzzle-specific insight involved |
| Segment, align, induce lexicon + word order, decode the question | **clingo** | this is the actual puzzle |
| Render the decoded frame as an English sentence | mechanical Python | inverse of gloss parsing |

The solver is given **no** Gilbertese-specific knowledge: no morpheme meanings, no word order,
no article/marker inventory. Everything is induced from the eight context pairs.

> Data provenance: pairs transcribed from the official PDF
> (`gs://cot-rosetta-interp-data/raw/uklo_pdf/www.uklo.org_wp-content_uploads_2023_03_2023_R1_3-Gilbertese.pdf`)
> and embedded below so the notebook is self-contained. The normalized copy lives at
> `gs://cot-rosetta-interp-data/puzzles/uklo-2023-r1-gilbertese.json`.

In [1]:
import re
from collections import defaultdict

import clingo

# UKLO 2023 R1 Problem 3, context pairs (verified against the official PDF).
# The puzzle states that English "fruit" is singular ('piece of fruit').
CONTEXT_PAIRS = [
    ("E nooria te uaa te aomata.",       "The man saw the fruit."),
    ("A na kunei taian uee taian aine.", "The women will find the flowers."),
    ("A na kunea te uee taian aomata.",  "The men will find the flower."),
    ("E na noorii taian uee te aine.",   "The woman will see the flowers."),
    ("A noorii taian uee taian aine.",   "The women saw the flowers."),
    ("E na kunei taian uee te aomata.",  "The man will find the flowers."),
    ("A kunei taian uee taian aine.",    "The women found the flowers."),
    ("A na kunea te uaa taian aine.",    "The women will find the fruit."),
]

QUESTION = "E noorii taian uaa te aomata."   # Q 3.1 (a): translate into English


def gil_tokens(sentence: str) -> list[str]:
    return re.sub(r"[.]", "", sentence).lower().split()


print("Context pairs:", len(CONTEXT_PAIRS))
print("Question tokens:", gil_tokens(QUESTION))

Context pairs: 8
Question tokens: ['e', 'noorii', 'taian', 'uaa', 'te', 'aomata']


## 1. The English interface (mechanical, not part of the puzzle)

Each English gloss is parsed into a flat semantic frame with six roles:
`subj_lemma, subj_num, tense, verb, obj_lemma, obj_num`. All glosses in this puzzle have the
shape *"The N (will) V the N."*, so a tiny deterministic parser suffices. The inverse
(frame → English) renders the solver's decoded answer. In the full Phase-4 pipeline an LLM
performs this grounding step; here it is ~30 lines of Python so that **everything
non-mechanical is done by the solver**.

In [2]:
NOUNS = {
    "man": ("man", "men"),
    "woman": ("woman", "women"),
    "fruit": ("fruit", "fruits"),
    "flower": ("flower", "flowers"),
}
VERBS = {"see": ("saw", "will see"), "find": ("found", "will find")}

NOUN_FORM = {
    form: (lemma, num)
    for lemma, (sg, pl) in NOUNS.items()
    for form, num in ((sg, "sg"), (pl, "pl"))
}
VERB_FORM = {}
for lemma, (past, fut) in VERBS.items():
    VERB_FORM[past] = (lemma, "past")
    VERB_FORM[fut.split()[-1]] = (lemma, "fut")   # bare form after 'will'


def parse_gloss(gloss: str) -> dict[str, str]:
    """'The women will find the fruit.' -> semantic frame (six roles)."""
    toks = re.sub(r"[.]", "", gloss).lower().split()
    assert toks[0] == "the", gloss
    i = 1
    subj_lemma, subj_num = NOUN_FORM[toks[i]]; i += 1
    tense = "past"
    if toks[i] == "will":
        tense = "fut"; i += 1
    verb, vtense = VERB_FORM[toks[i]]; i += 1
    assert tense == "fut" or vtense == "past", gloss
    assert toks[i] == "the", gloss
    i += 1
    obj_lemma, obj_num = NOUN_FORM[toks[i]]
    return {
        "subj_lemma": subj_lemma, "subj_num": subj_num, "tense": tense,
        "verb": verb, "obj_lemma": obj_lemma, "obj_num": obj_num,
    }


def gen_english(frame: dict[str, str]) -> str:
    """Semantic frame -> English sentence (inverse of parse_gloss)."""
    subj = NOUNS[frame["subj_lemma"]][frame["subj_num"] == "pl"]
    obj = NOUNS[frame["obj_lemma"]][frame["obj_num"] == "pl"]
    verb = VERBS[frame["verb"]][0 if frame["tense"] == "past" else 1]
    return f"The {subj} {verb} the {obj}."


for gil, eng in CONTEXT_PAIRS:
    frame = parse_gloss(eng)
    assert gen_english(frame) == eng, (gen_english(frame), eng)   # round-trip check
    print(f"{gil:36} {frame}")

E nooria te uaa te aomata.           {'subj_lemma': 'man', 'subj_num': 'sg', 'tense': 'past', 'verb': 'see', 'obj_lemma': 'fruit', 'obj_num': 'sg'}
A na kunei taian uee taian aine.     {'subj_lemma': 'woman', 'subj_num': 'pl', 'tense': 'fut', 'verb': 'find', 'obj_lemma': 'flower', 'obj_num': 'pl'}
A na kunea te uee taian aomata.      {'subj_lemma': 'man', 'subj_num': 'pl', 'tense': 'fut', 'verb': 'find', 'obj_lemma': 'flower', 'obj_num': 'sg'}
E na noorii taian uee te aine.       {'subj_lemma': 'woman', 'subj_num': 'sg', 'tense': 'fut', 'verb': 'see', 'obj_lemma': 'flower', 'obj_num': 'pl'}
A noorii taian uee taian aine.       {'subj_lemma': 'woman', 'subj_num': 'pl', 'tense': 'past', 'verb': 'see', 'obj_lemma': 'flower', 'obj_num': 'pl'}
E na kunei taian uee te aomata.      {'subj_lemma': 'man', 'subj_num': 'sg', 'tense': 'fut', 'verb': 'find', 'obj_lemma': 'flower', 'obj_num': 'pl'}
A kunei taian uee taian aine.        {'subj_lemma': 'woman', 'subj_num': 'pl', 'tense': 'past', 'verb'

## 2. The grammar-induction encoding

The hypothesis space mirrors the canonical human solver steps from our CoT taxonomy
(*segment → align → hypothesize → test-against-pair*):

- **Sentence elements.** Every frame induces seven abstract elements, one per syntactic slot:
  subject-number marker, tense marker, verb form, object article, object noun, subject article,
  subject noun. Articles share one *category* across the two positions, and so do nouns — this is
  the **one form ↔ one meaning** inductive bias a human solver applies.
- **Lexicon hypothesis (choice rule).** Each (category, value) pair is realized as exactly one
  surface word — or as *zero* (silence), which is how the solver can discover that past tense is
  unmarked.
- **Word-order hypothesis (choice rule).** A strict total order over the seven slots.
- **Test against every pair (integrity constraints).** For each sentence, the non-zero elements,
  sorted by slot rank, must reproduce the token sequence *exactly* — every token accounted for,
  no token left over. One constraint set, applied to all eight context pairs **and** the question.
- **Decoding as abduction.** The question sentence's frame is unknown, so it is itself a choice
  rule over frames. Any answer set must therefore contain a frame for the question that the
  induced grammar generates as exactly the observed tokens — solving the puzzle and translating
  the question are one joint satisfiability problem.

Enumerating **all** answer sets tells us whether the eight pairs pin down a unique grammar and a
unique reading of the question.

**Where did this hypothesis space come from?** The scaffolding — choice rules for a lexicon and
an order, integrity constraints demanding exact regeneration — is the standard ASP
*generate-and-test* recipe for induction problems and is puzzle-independent. The
puzzle-*specific* design decisions were made by eyeballing the data shape (not the solution):
the English glosses all fit one frame with six roles; sentences have 6–7 tokens against 6
frame roles, suggesting up to seven surface elements with one optional; verb surface forms vary
within a lemma (`nooria`/`noorii`), so the verb element is keyed by the lemma *plus one*
nominal feature and the solver decides whether that dependency is real. Nothing
answer-shaped is smuggled in: which word means what, which slot goes where, which element is
silent, and whether verb agreement exists at all are left to the solver. Designing this space
is exactly the role the LLM plays in the full Arm-C pipeline; here a human did it once by hand.

In [3]:
ENCODING = r"""
% ---------- hypothesis space for the question sentence's frame ----------
num(sg;pl).  tns_val(past;fut).
nounlemma(L) :- csem(_,subj_lemma,L).
nounlemma(L) :- csem(_,obj_lemma,L).
verblemma(V) :- csem(_,verb,V).

1 { sem(S,subj_lemma,L) : nounlemma(L) } 1 :- qsent(S).
1 { sem(S,obj_lemma,L)  : nounlemma(L) } 1 :- qsent(S).
1 { sem(S,subj_num,N)   : num(N)       } 1 :- qsent(S).
1 { sem(S,obj_num,N)    : num(N)       } 1 :- qsent(S).
1 { sem(S,tense,T)      : tns_val(T)   } 1 :- qsent(S).
1 { sem(S,verb,V)       : verblemma(V) } 1 :- qsent(S).
sem(S,R,V) :- csem(S,R,V).            % context frames are given

% ---------- sentence elements: slot / category / value ----------
% Articles share category `art`, nouns share `nn`: one form <-> one meaning
% regardless of subject vs object position.
el(S,marker, mrk, N)      :- sem(S,subj_num,N).
el(S,tense_m,tns, T)      :- sem(S,tense,T).
el(S,verb_m, vrb, f(V,N)) :- sem(S,verb,V), sem(S,obj_num,N).
el(S,art_o,  art, N)      :- sem(S,obj_num,N).
el(S,noun_o, nn,  L)      :- sem(S,obj_lemma,L).
el(S,art_s,  art, N)      :- sem(S,subj_num,N).
el(S,noun_s, nn,  L)      :- sem(S,subj_lemma,L).

% ---------- lexicon hypothesis: (category,value) -> word or zero ----------
cv(C,V) :- el(_,_,C,V).
word(W) :- tok(_,_,W).
1 { real(C,V,W) : word(W) ; real(C,V,zero) } 1 :- cv(C,V).

% ---------- word-order hypothesis: strict total order over slots ----------
slotname(marker;tense_m;verb_m;art_o;noun_o;art_s;noun_s).
1 { rank(Sl,R) : R=1..7 } 1 :- slotname(Sl).
:- rank(Sl1,R), rank(Sl2,R), Sl1 < Sl2.

% ---------- test against every sentence: generate exactly the tokens ----------
nz(S,Sl) :- el(S,Sl,C,V), not real(C,V,zero).
pos(S,Sl,N+1) :- nz(S,Sl), rank(Sl,R),
                 N = #count{ Sl2 : nz(S,Sl2), rank(Sl2,R2), R2 < R }.
:- pos(S,Sl,P), el(S,Sl,C,V), real(C,V,W), W != zero, not tok(S,P,W).
:- sent(S), N1 = #count{ P : tok(S,P,_) }, N2 = #count{ Sl : nz(S,Sl) }, N1 != N2.

#show real/3.
#show rank/2.
#show sem/3.
"""

def corpus_facts(context_pairs, question) -> str:
    """Encode the puzzle instance as ASP facts."""
    lines = []
    for i, (gil, eng) in enumerate(context_pairs, 1):
        s = f"s{i}"
        lines.append(f"sent({s}).")
        for p, w in enumerate(gil_tokens(gil), 1):
            lines.append(f'tok({s},{p},"{w}").')
        for role, val in parse_gloss(eng).items():
            lines.append(f"csem({s},{role},{val}).")
    lines.append("sent(q). qsent(q).")
    for p, w in enumerate(gil_tokens(question), 1):
        lines.append(f'tok(q,{p},"{w}").')
    return "\n".join(lines)

print(corpus_facts(CONTEXT_PAIRS[:1], QUESTION))

sent(s1).
tok(s1,1,"e").
tok(s1,2,"nooria").
tok(s1,3,"te").
tok(s1,4,"uaa").
tok(s1,5,"te").
tok(s1,6,"aomata").
csem(s1,subj_lemma,man).
csem(s1,subj_num,sg).
csem(s1,tense,past).
csem(s1,verb,see).
csem(s1,obj_lemma,fruit).
csem(s1,obj_num,sg).
sent(q). qsent(q).
tok(q,1,"e").
tok(q,2,"noorii").
tok(q,3,"taian").
tok(q,4,"uaa").
tok(q,5,"te").
tok(q,6,"aomata").


## 3. Solve: induce the grammar and decode the question in one shot

In [4]:
def solve(context_pairs, question):
    """Return every answer set (as lists of atom strings)."""
    ctl = clingo.Control(["0"])                    # 0 = enumerate ALL models
    ctl.add("base", [], corpus_facts(context_pairs, question) + ENCODING)
    ctl.ground([("base", [])])
    models = []
    with ctl.solve(yield_=True) as handle:
        for model in handle:
            models.append([str(a) for a in model.symbols(shown=True)])
    return models


def question_frame(atoms: list[str]) -> dict[str, str]:
    frame = {}
    for a in atoms:
        m = re.match(r"sem\(q,(\w+),(\w+)\)", a)
        if m:
            frame[m.group(1)] = m.group(2)
    return frame


models = solve(CONTEXT_PAIRS, QUESTION)
frames = {tuple(sorted(question_frame(m).items())) for m in models}
print(f"answer sets: {len(models)}   distinct question readings: {len(frames)}")
assert len(frames) == 1, "context pairs do not pin down a unique reading"
frame = dict(next(iter(frames)))
print("decoded frame:", frame)

answer sets: 1   distinct question readings: 1
decoded frame: {'obj_lemma': 'fruit', 'obj_num': 'pl', 'subj_lemma': 'man', 'subj_num': 'sg', 'tense': 'past', 'verb': 'see'}


In [5]:
CAT_LABEL = {
    "mrk": "subject-number marker",
    "tns": "tense marker",
    "vrb": "verb form  f(lemma, object number)",
    "art": "article",
    "nn": "noun",
}

atoms = models[0]
print("=== Induced constituent order ===")
ranks = sorted(
    (int(m.group(2)), m.group(1))
    for m in (re.match(r"rank\((\w+),(\d+)\)", a) for a in atoms) if m
)
print("  " + "  <  ".join(slot for _, slot in ranks))

print("\n=== Induced lexicon ===")
by_cat = defaultdict(list)
for a in atoms:
    m = re.match(r'real\((\w+),(.+),(?:"(.+)"|(zero))\)', a)
    if m:
        by_cat[m.group(1)].append((m.group(2), m.group(3) or "∅ (unmarked)"))
for cat in ("mrk", "tns", "vrb", "art", "nn"):
    print(f"  {CAT_LABEL[cat]}:")
    for value, surface in sorted(by_cat[cat]):
        print(f"    {value:14} -> {surface}")

=== Induced constituent order ===
  marker  <  tense_m  <  verb_m  <  art_o  <  noun_o  <  art_s  <  noun_s

=== Induced lexicon ===
  subject-number marker:
    pl             -> a
    sg             -> e
  tense marker:
    fut            -> na
    past           -> ∅ (unmarked)
  verb form  f(lemma, object number):
    f(find,pl)     -> kunei
    f(find,sg)     -> kunea
    f(see,pl)      -> noorii
    f(see,sg)      -> nooria
  article:
    pl             -> taian
    sg             -> te
  noun:
    flower         -> uee
    fruit          -> uaa
    man            -> aomata
    woman          -> aine


In [6]:
answer = gen_english(frame)
print(f'Q 3.1 (a)  "{QUESTION}"')
print(f"answer  ->  {answer}")

OFFICIAL = "The man saw the fruits."           # UKLO mark scheme, 2023 R1 P3
assert answer == OFFICIAL
print("matches the official UKLO mark scheme ✓")

Q 3.1 (a)  "E noorii taian uaa te aomata."
answer  ->  The man saw the fruits.
matches the official UKLO mark scheme ✓


### Counterfactual check: every rival reading is refuted

Getting one consistent reading is only half the story — the value of a solver is that it can
*prove* the alternatives wrong. Here we force each rival hypothesis about the question's frame
(via an extra integrity constraint) and re-solve. A reading is refuted when the program becomes
unsatisfiable: **no** grammar consistent with all eight context pairs can generate the question
sentence under that reading.

In [7]:
def count_models_forcing(constraint: str) -> int:
    """Number of answer sets when an extra constraint pins part of q's frame."""
    ctl = clingo.Control(["0"])
    ctl.add("base", [], corpus_facts(CONTEXT_PAIRS, QUESTION) + ENCODING + constraint)
    ctl.ground([("base", [])])
    n = 0
    with ctl.solve(yield_=True) as handle:
        for _ in handle:
            n += 1
    return n


COUNTERFACTUALS = [
    (":- not sem(q,obj_num,pl).",       "obj_num = pl   ('the fruits' — the answer)",
     "—"),
    (":- not sem(q,obj_num,sg).",       "obj_num = sg   ('the fruit')",
     "verb would have to be f(see,sg) = nooria, token is noorii"),
    (":- not sem(q,tense,fut).",        "tense = fut    ('will see')",
     "na would need a 7th token; sentence has 6"),
    (":- not sem(q,subj_lemma,woman).", "subj = woman   ('the woman saw ...')",
     "subject-noun token is aomata, but nn(woman) = aine"),
    (":- not sem(q,subj_num,pl).",      "subj_num = pl  ('the men saw ...')",
     "marker would have to be a, token 1 is e"),
    (":- not sem(q,verb,find).",        "verb = find    ('found')",
     "form would have to be kunea/kunei, token is noorii"),
]

print(f"{'forced reading':46} {'models':>6}   refutation")
for constraint, reading, why in COUNTERFACTUALS:
    n = count_models_forcing(constraint)
    verdict = "consistent" if n else f"UNSAT — {why}"
    print(f"{reading:46} {n:>6}   {verdict}")

assert [count_models_forcing(c) for c, _, _ in COUNTERFACTUALS] == [1, 0, 0, 0, 0, 0]

forced reading                                 models   refutation


obj_num = pl   ('the fruits' — the answer)          1   consistent


obj_num = sg   ('the fruit')                        0   UNSAT — verb would have to be f(see,sg) = nooria, token is noorii


tense = fut    ('will see')                         0   UNSAT — na would need a 7th token; sentence has 6


subj = woman   ('the woman saw ...')                0   UNSAT — subject-noun token is aomata, but nn(woman) = aine


subj_num = pl  ('the men saw ...')                  0   UNSAT — marker would have to be a, token 1 is e


verb = find    ('found')                            0   UNSAT — form would have to be kunea/kunei, token is noorii


So the decoded answer is not merely *a* consistent reading: every alternative value for every
role is provably inconsistent with the context pairs. This is the "test-against-pair" solver
step made exhaustive — an LLM's chain of thought can assert this kind of claim, the solver
establishes it.

The induced grammar is exactly the one in the official mark-scheme commentary — discovered by
the solver, not supplied to it:

- word order **marker < tense < verb < object article < object noun < subject article < subject noun** (VOS),
- `e` = singular-subject marker, `a` = plural-subject marker,
- `na` = future, past = **zero** (the solver hypothesized silence and it survived all eight pairs),
- `te` = singular article, `taian` = plural article,
- verb surface form depends on the **object's** number — the puzzle's key twist, visible in the
  induced `f(lemma, object-number)` table before we even segment the verbs.

The decode itself is the interesting part: `E … te aomata` forces a singular subject, so
`taian uaa` can only be the *object* — number agreement and word order jointly disambiguate,
which is precisely the contextual inference chain a human solver verbalizes.

## 4. Robustness: leave-one-out

Hold out each context pair in turn, induce the grammar from the remaining seven, and decode the
held-out Gilbertese sentence blind. This checks that the induction genuinely generalizes rather
than memorizing.

In [8]:
for i, (held_gil, held_eng) in enumerate(CONTEXT_PAIRS, 1):
    rest = CONTEXT_PAIRS[:i-1] + CONTEXT_PAIRS[i:]
    loo_frames = {tuple(sorted(question_frame(m).items())) for m in solve(rest, held_gil)}
    preds = sorted({gen_english(dict(f)) for f in loo_frames})
    status = "✓" if preds == [held_eng] else f"✗ predicted {preds}"
    print(f"  pair {i}: {held_eng:36} {status}")

  pair 1: The man saw the fruit.               ✓


  pair 2: The women will find the flowers.     ✓


  pair 3: The men will find the flower.        ✓


  pair 4: The woman will see the flowers.      ✓


  pair 5: The women saw the flowers.           ✓


  pair 6: The man will find the flowers.       ✓


  pair 7: The women found the flowers.         ✓


  pair 8: The women will find the fruit.       ✓


## 5. Bonus: morphological segmentation of the verb forms

The main model treats each verb form as an opaque function of (lemma, object number). A human
solver goes one step further and *segments*: `nooria = noori + a`. A second tiny ASP program
recovers stems and suffixes from the induced verb-form table, under the constraints *stem is a
function of the lemma* and *suffix is a function of the object's number*.

In [9]:
verb_forms = {}
for a in atoms:
    m = re.match(r'real\(vrb,f\((\w+),(\w+)\),"(\w+)"\)', a)
    if m:
        verb_forms[(m.group(1), m.group(2))] = m.group(3)

seg_facts = []
for (lemma, number), form in verb_forms.items():
    seg_facts.append(f"vform({lemma},{number}).")
    for i in range(1, len(form) + 1):            # stem non-empty, suffix may be empty
        seg_facts.append(f'cand({lemma},{number},"{form[:i]}","{form[i:]}").')

SEGMENT = "\n".join(seg_facts) + r"""
1 { pick(L,N,Stem,Suf) : cand(L,N,Stem,Suf) } 1 :- vform(L,N).
:- pick(L,_,Stem1,_), pick(L,_,Stem2,_), Stem1 != Stem2.  % stem = f(lemma)
:- pick(_,N,_,Suf1),  pick(_,N,_,Suf2),  Suf1 != Suf2.    % suffix = f(object number)
#show pick/4.
"""

ctl = clingo.Control(["0"])
ctl.add("base", [], SEGMENT)
ctl.ground([("base", [])])
seg_models = []
with ctl.solve(yield_=True) as handle:
    for model in handle:
        seg_models.append(sorted(str(a) for a in model.symbols(shown=True)))

print(f"segmentations consistent with both constraints: {len(seg_models)}")
for a in seg_models[0]:
    m = re.match(r'pick\((\w+),(\w+),"(\w+)","(\w+)"\)', a)
    lemma, number, stem, suffix = m.groups()
    print(f"  {lemma:5} + {number}-object  =  {stem}- + -{suffix}")

segmentations consistent with both constraints: 1
  find  + pl-object  =  kune- + -i
  find  + sg-object  =  kune- + -a
  see   + pl-object  =  noori- + -i
  see   + sg-object  =  noori- + -a


Unique segmentation: stems `noori-` 'see' and `kune-` 'find', object-agreement suffixes
`-a` (singular) / `-i` (plural) — again matching the official commentary.

## 6. Takeaways for Phase 4

- **What the solver did:** from 8 pairs it induced the full lexicon, the VOS constituent order,
  zero-marked past tense, and object-number agreement; certified the grammar and the decode of
  the question as *unique*; and went 8/8 on leave-one-out. Total solve time: milliseconds, no
  model calls, fully deterministic.
- **What the solver did *not* do:** design the hypothesis space. The slot/category structure
  above fits this puzzle's sentence shape; a different puzzle (case systems, reduplication,
  non-concatenative morphology) needs a different — though similarly compact — space. That is
  exactly the division of labor for **Arm C** of the Phase-4 design: the LLM proposes the
  hypothesis space (as ASP rules), clingo does the exhaustive search-and-verify, and
  counterexamples flow back to the LLM.
- **Why this beats monolithic CoT on its home turf:** the answer comes with a proof of
  uniqueness against all context pairs. An LLM's CoT can *assert* "this is consistent with all
  the data"; the solver *establishes* it. Conversely, the English-side grounding (here trivial)
  is where LLMs are irreplaceable — puzzles whose glosses need real semantic work (kinship,
  deixis, free translation) will not reduce to a 30-line parser.
- **Next:** wrap this pattern as a reusable harness (`puzzle JSON -> facts`, LLM-proposed
  encoding, clingo verify), then run Arms A/B/C on the Phase-1 pilot set and break results out
  by `phenomena` tags.